In [3]:
import torch
from torch.utils.data import DataLoader
from dataloader.llm_dataset import LLMDataset

In [4]:
# Read the raw text from the file
with open("../data/the-verdict.txt", "r") as file:
    raw_text = file.read()

# Split into training and validation sets
split_idx = int(0.85 * len(raw_text))
train_text = raw_text[:split_idx]
val_text = raw_text[split_idx:]

# Create a dataset
train_dataset = LLMDataset(context_length=256, stride=8, raw_text=train_text)
val_dataset = LLMDataset(context_length=256, stride=8, raw_text=val_text)

# Create a dataloader for the dataset
train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, drop_last=True)
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False, drop_last=True)

In [5]:
# Print the lengths of the training and validation sets
print(f"Length of training set: {len(train_dataset)}")
print(f"Length of validation set: {len(val_dataset)}")

Length of training set: 513
Length of validation set: 67


In [6]:
# Get the device to run the model on
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
# Instantiate the GPTModel 
from architecture.gpt_model import GPTModel
from architecture.model_config import cfg_124m

gpt_model = GPTModel(cfg_124m)
gpt_model.to(device)

print(f"Model instantiated and moved to device: {device}")

Model instantiated and moved to device: cuda


In [8]:
# Create an optimizer for the model
import torch.optim as optim
optimizer = optim.AdamW(gpt_model.parameters(), lr=cfg_124m["learning_rate"])

# Create a loss criterion
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
## Setup W&B
import wandb

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="mechtrics7-task-proxemics",
    # Set the wandb project where this run will be logged.
    project="llm_scratch",
    # Track hyperparameters and run metadata.
    config=cfg_124m
)

# Create a training loop for the model
NUM_EPOCHS = 10
VAL_FREQUENCY = 2  # Example value, adjust as needed

# Set the model to training mode
gpt_model.train()

# List to store the loss values for each epoch
train_epoch_losses = []
val_epoch_losses = []
step_losses = []

for epoch in range(NUM_EPOCHS):
    print(f"Starting epoch {epoch+1}/{NUM_EPOCHS}")

    for batch_idx, (inputs, targets) in enumerate(train_dataloader):
        # Move inputs and targets to the device
        # Inputs shape: (batch_size, context_length)
        # Targets shape: (batch_size, context_length)
        inputs, targets = inputs.to(device), targets.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        # outputs shape: (batch_size, context_length, vocab_size)
        outputs = gpt_model(inputs)

        # Compute the loss
        loss = criterion(outputs.view(-1, outputs.size(-1)), targets.view(-1))

        # Backward pass and optimization step
        loss.backward()
        optimizer.step()

        step_losses.append(loss.item())

        run.log({"step_loss": loss.item()})

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {loss.item():.4f}")
    train_epoch_losses.append(loss.item())
    run.log({"train_epoch_loss": loss.item()})

    # Perform validation at the specified frequency
    if (epoch + 1) % VAL_FREQUENCY == 0:
        gpt_model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for val_inputs, val_targets in val_dataloader:
                val_inputs, val_targets = val_inputs.to(device), val_targets.to(device)
                val_outputs = gpt_model(val_inputs)
                loss = criterion(val_outputs.view(-1, val_outputs.size(-1)), val_targets.view(-1))
                val_loss += loss.item()
        val_loss /= len(val_dataloader)
        val_epoch_losses.append(val_loss)
        print(f"Validation Loss after Epoch [{epoch+1}/{NUM_EPOCHS}]: {val_loss:.4f}")
        run.log({"val_loss": val_loss})
        gpt_model.train()

# Print the average loss for the epoch
avg_loss = sum(train_epoch_losses[-len(train_dataloader):]) / len(train_dataloader)
print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Average Loss: {avg_loss:.4f}")

# Set the model back to evaluation mode after training
gpt_model.eval()
print("Training complete. Model is now in evaluation mode.")

# Save the trained model checkpoint locally
torch.save(gpt_model.state_dict(), "ckpt/gpt_model_checkpoint_expt_1.pth")

# Save the trained model checkpoint to the W&B run as an artifact
artifact = wandb.Artifact("gpt_model_checkpoint", type="model")
artifact.add_file("ckpt/gpt_model_checkpoint_expt_1.pth")
run.log_artifact(artifact)

# Finish the W&B run
run.finish()

In [ ]:
# Plot the training step losses
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(step_losses, label="Step Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training Step Loss")
plt.legend()
plt.grid(True)
plt.show()

In [15]:
# Try out the model: 
import tiktoken
from utils import generate_sequence_with_temperature_and_top_k

import torch

torch.manual_seed(254)

# Load the GPT model and set it to evaluation mode
gpt_model.load_state_dict(torch.load("ckpt/gpt_model_checkpoint_expt_1.pth"))
gpt_model.eval()

# Set the temperature and top-k sampling parameters
temperature = 0.1
top_k = 3

# Define the starting prompt for text generation
start_prompt_text = "Once upon a time in a Kingdom far away"

# Encode using the gpt-2 tokenizer
encoding = tiktoken.get_encoding("gpt2")
start_prompt_ids = encoding.encode(start_prompt_text)

print("Input IDs:", start_prompt_ids)

# Generate a sequence using the model
output_ids = generate_sequence_with_temperature_and_top_k(gpt_model, start_prompt_ids, max_length=25, device=device, temperature=temperature, top_k=top_k)

# Decode the output IDs back to text
output_text = encoding.decode(output_ids)
print("Output Text:", output_text)

Input IDs: [7454, 2402, 257, 640, 287, 257, 7526, 1290, 1497]
Output Text: Once upon a time in a Kingdom far away, the last word of Jack's the more modestmate skill, the tricks of detail," Mrs. Gisburn said with
